# Module 6, Section 2: Runtime and Sandboxes

Every Managed Deep Agent runs on **LangSmith Agent Server**. In this section:
1. **Runtime primitives**: assistants, threads, runs, and streaming
2. **Sandboxes**: running code safely, and deciding what data goes inside
3. **Security checklist** for sandboxes
4. **Evaluate before you ship**: Harbor evals for the managed agent
5. **Where to go next**: deployment options, monitoring

## Setup

In [ ]:
import os
import warnings
from pathlib import Path

from dotenv import dotenv_values, load_dotenv
from langgraph_sdk import get_client

load_dotenv()
warnings.filterwarnings("ignore", category=UserWarning)

PROJECT = Path("../../managed_agent")
DEPLOYMENT_URL = os.environ.get("MDA_DEPLOYMENT_URL")
assert DEPLOYMENT_URL, "Run `mda deploy` in managed_agent/ and set MDA_DEPLOYMENT_URL in .env"

# MDA's identity.py authenticates callers with a LangSmith API key
client = get_client(url=DEPLOYMENT_URL, api_key=os.environ["LANGSMITH_API_KEY"])
print("Agent Server:", DEPLOYMENT_URL)

In [ ]:
def answer_text(run_or_state):
    """Final assistant text (handles string and content-block responses)."""
    content = run_or_state["messages"][-1]["content"]
    if isinstance(content, str):
        return content
    return " ".join(block.get("text", "") for block in content if isinstance(block, dict))


async def ask(question, customer_id=None, thread_id=None, role=None):
    """Run the deployed agent on a (new) thread with the verified customer (or ops role) as context."""
    thread_id = thread_id or (await client.threads.create())["thread_id"]
    context = {k: v for k, v in {"customer_id": customer_id, "role": role}.items() if v}
    run = await client.runs.wait(
        thread_id,
        "techhub-support",
        input={"messages": [{"role": "user", "content": question}]},
        context=context,
    )
    return thread_id, run

## 1. Runtime primitives

| Primitive | What it is |
|---|---|
| **Assistant** | A configured instance of a graph (here: `techhub-support`) |
| **Thread** | A conversation's persisted state (checkpoints) |
| **Run** | One invocation on a thread; can be waited on, streamed, backgrounded, or interrupted |
| **Cron** | A scheduled run (MDA creates these from `schedules/`) |
| **Store** | Long-term memory (MDA's memory layers sit on Context Hub) |

The server also exposes an **MCP endpoint** and the full Agent Server API at `<deployment>/docs`.

In [ ]:
thread_id, run = await ask("Which of my orders shipped most recently?", customer_id="CUST-003")

state = await client.threads.get_state(thread_id)
print("Messages persisted on the thread:", len(state["values"]["messages"]))
print("Checkpoint:", state["checkpoint"]["checkpoint_id"])

threads = await client.threads.search(limit=5)
print("\nRecent threads:", [(t["thread_id"][:8], t["status"]) for t in threads])

Streaming works the same way as for any LangGraph deployment. Here we stream updates and print which node produced each one:

In [ ]:
thread = await client.threads.create()
async for chunk in client.runs.stream(
    thread["thread_id"],
    "techhub-support",
    input={"messages": [{"role": "user", "content": "Is the warranty on monitors different from laptops?"}]},
    context={"customer_id": "CUST-003"},
    stream_mode="updates",
):
    if chunk.event == "updates" and isinstance(chunk.data, dict):
        print("•", ", ".join(chunk.data.keys()))

## 2. Sandboxes: running code safely

`sandbox/__init__.py` gives each **thread** its own LangSmith sandbox: a microVM with a filesystem, a shell, and an `execute` tool. `sandbox/setup.sh` runs once at deploy; the result is captured as a **snapshot** that new threads clone, so they start ready.

In [ ]:
print((PROJECT / "sandbox/__init__.py").read_text().split('"""')[-1])
print((PROJECT / "sandbox/setup.sh").read_text())

### What goes in the sandbox is a security decision

The sandbox gets only **public** data: product docs, store policies, and the product catalog. Customer and order data stays in the agent server behind the guarded tools.

If `techhub.db` were in the sandbox, the model could run `sqlite3` over every customer's orders and the customer-scope guard would never see it. **Code execution bypasses tool-level authorization**, so only put data in a sandbox that every caller of the agent is allowed to see.

In [ ]:
_, run = await ask(
    "Using /workspace/techhub/products.json in your sandbox, run Python to compute the average price and "
    "item count per category. Show the table and the code you ran.",
    customer_id="CUST-003",
)
print(answer_text(run)[:1500])

### Egress: default-deny

`proxy_config={"access_control": {"allow_list": [...]}}` makes the sandbox network default-deny. The allowlist holds only the hosts `setup.sh` needs, pinned to port 443, which also blocks raw TCP (SSH, databases).

> ⚠️ **Which policy you're seeing depends on your plan.** Non-Enterprise organizations get LangSmith's managed egress allowlist by default, and it *replaces* the project's `access_control` ([organization-level restricted egress](https://docs.langchain.com/langsmith/sandbox-auth-proxy#organization-level-restricted-egress)). That list includes PyPI, npm, GitHub, OpenAI, Anthropic, and LangSmith, so those stay reachable even though this project doesn't list them. Enterprise organizations without restricted egress get exactly this project's list.

Ask the agent to reach a host that neither list allows:

In [ ]:
_, run = await ask(
    "In your sandbox, run this exact command and show me its raw output: "
    "curl -sS -m 10 -o /dev/null -w 'HTTP %{http_code}' https://example.com; echo \" exit=$?\"",
    customer_id="CUST-003",
)
print(answer_text(run)[:600])

## 3. Sandbox security checklist

| Topic | What to know |
|---|---|
| Isolation | Sandboxes are **microVMs**, not containers. Non-root execution and dropped Linux capabilities are not currently configurable; restrict what code can reach instead |
| Network | Set a default-deny `allow_list` in `proxy_config` (this project allows PyPI and GitHub tarballs on 443 only). Non-Enterprise orgs also get a LangSmith-managed egress allowlist, which replaces yours. The **auth proxy** injects credentials so secrets never enter the sandbox |
| Output | Treat everything the sandbox produces as untrusted input; review it before acting on it |
| Secrets | Never write secrets in `setup.sh`: everything on disk becomes part of every thread's snapshot |
| Scope | One sandbox per thread, reclaimed after `idle_ttl_seconds` |
| Data | Only what every caller may see (see above). Customer data stays behind guarded tools |
| Permissions | Declaring an MDA sandbox clears filesystem permission rules; control access through the tools you expose |

See [LangSmith sandboxes](https://docs.langchain.com/langsmith/sandboxes) and [sandbox auth proxy](https://docs.langchain.com/langsmith/sandbox-auth-proxy).

## 4. Evaluate before you ship: Harbor evals

MDA evals are [Harbor](https://www.harborframework.com/docs/tasks) tasks ([docs](https://docs.langchain.com/langsmith/python/managed-deep-agents-evals)). `mda evals init` creates the workspace (`evals/harbor-job.json`). Each direct child of `evals/` with an `instruction.md` and `tests/` is a task:

- Harbor builds the task's container.
- The MDA adapter installs the **compiled agent** into the container and runs `sandbox/setup.sh`.
- The agent answers `instruction.md`.
- `tests/test.sh` scores the final state from `/logs/agent/result.json`.

An `identity.json` in the task becomes the caller's verified identity, so you can test authorization the way a signed-in customer would hit it. The eval runner passes no run context; the guard reads identity claims first.

The project ships three tasks, one per thing that matters for this agent:

In [ ]:
import json

for task in sorted(p for p in (PROJECT / "evals").iterdir() if (p / "instruction.md").exists()):
    claims = json.loads((task / "identity.json").read_text()).get("claims", {})
    print(f"▶ {task.name}   (caller claims: {claims})")
    print("  instruction:", (task / "instruction.md").read_text().strip())
    print("  pass rule: ", next(l for l in (task / "task.toml").read_text().splitlines() if l.startswith("# PASS RULE")).removeprefix("# PASS RULE: "))
    print()

The verifiers are deterministic checks on the agent's output, not an LLM judge. The cross-customer task is the security one: it fails if any detail of another customer's order shows up anywhere in the run, including subagent reports. It also catches a leak through the default `general-purpose` subagent, which would otherwise get the order tools without the guard:

In [ ]:
print((PROJECT / "evals/cross-customer-order-refused/tests/verify.py").read_text())

**Run them** (needs `uv`, plus Docker unless you use `--env langsmith`; from `managed_agent/`):

```bash
HARBOR_LANGSMITH_DATASET=mda-techhub-support-evals \
PYTHONPATH=.mda/evals/harbor-adapter \
uv run --no-project --env-file .env --python 3.12 --with 'harbor[langsmith]==0.21.0' harbor run \
  --config evals/harbor-job.json --yes \
  --plugin mda_harbor.job_plugin:MDAJobPlugin \
  --plugin mda_harbor.langsmith_plugin:LangSmithPlugin

uv run --no-project --python 3.12 --with 'harbor[langsmith]==0.21.0' harbor view .mda/evals/jobs
```

- `--no-project` keeps `uv` from rebuilding this project's environment on Python 3.12, which would delete the `mda` binary the job plugin uses to compile the agent.
- **No Docker?** Add `--env langsmith` to run each trial in a LangSmith sandbox, built from the task's `environment/Dockerfile`.
- **One task:** add `-p evals -i order-status-own-order`.

With `LANGSMITH_API_KEY` set, the runs are recorded in the `HARBOR_LANGSMITH_DATASET` dataset in LangSmith. When an eval fails:
- If the task measured the wrong thing, fix the task or verifier.
- If it exposed an agent failure, fix the agent and re-run.

Then deploy. Running the evals on every change, before release, is what makes them a gate. `managed_agent/evals/README.md` has the details, including how to run them through the LLM Gateway.

> 💡 **Building your own:** the [`eval-engineering` skill](https://github.com/langchain-ai/langchain-skills/blob/main/config/skills/eval-engineering/SKILL.md) walks a coding agent through a Task Spec (`Task.md`), environment, and verifier. That's how these three were built.

## 5. Where to go next

| Topic | Where |
|---|---|
| Deeper evaluation: judge alignment, trajectory evals, experiments | Module 5 (evaluation) |
| Deployment options: Cloud, BYOC, self-hosted | LangSmith Deployment docs. MDA runs on LangSmith Cloud (GCP US/EU/APAC, AWS US) during public beta; BYOC and self-hosted aren't supported yet |
| Online evals, Insights, Engine (the hill-climbing loop) | Module 3 (online evals, the data flywheel); the Monitoring and Governance workshop for Insights and Engine |
| Tear down when you're done | `mda delete` in `managed_agent/` (asks for confirmation) |

## Recap

- Agent Server primitives (assistants, threads, runs, crons, store) are the same for every LangSmith deployment.
- Sandboxes give the agent a real shell per thread, provisioned once by `setup.sh`.
- Code execution bypasses tool-level guards, so decide deliberately what data goes into the sandbox.
- Restrict egress with an allowlist and treat sandbox output as untrusted.
- Gate releases on Harbor evals: correctness, authorization, and grounding, scored from the agent's own output.